> 📄 English: [web-agent-tutorial.ipynb](web-agent-tutorial.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--web-agent-tutorial)

# 2. 构建 Web 研究 Agent 🧙

欢迎来到教程 2！

在第一个练习中，你学习了
Tavily API 的基础知识。现在，让我们更进一步：我们将
这些知识与 LLM 结合起来，释放
真正的价值。在本教程中，你将学习如何构建一个能够对实时 Web 数据进行搜索、提取、抓取和推理的 Web 研究 Agent。

你已经探索了每个 Tavily API 端点可用的各种参数
配置。借助官方 [Tavily-LangChain 集成](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--web-agent-tutorial&click=tavily-langchain&target=https%3A%2F%2Fwww.tavily.com%2Fintegrations%2Flangchain&text=Tavily-LangChain%20integration)，我们的 Agent 可以
根据每个任务的上下文和需求，自动设置这些参数——比如搜索的
`time_range` 或特定的 crawl
`instructions`。这种由 LLM 驱动的动态配置在
Agent 系统中非常强大。

学完本课，你将知道如何：
- 将基础模型无缝连接到 Web，以进行最新研究
- 构建一个 react 风格的 Web Agent
- 通过 Tavily-LangChain 集成动态配置 search、extract 和 crawl 参数。



---


## 快速开始

请按照以下步骤配置：

1. **注册** Tavily：访问 [app.tavily.com](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--web-agent-tutorial&click=tavily-app&target=https%3A%2F%2Fapp.tavily.com%2Fhome%2F%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=app.tavily.com) 获取你的 API key。

   *可参考下方链接的截图，获取分步指引：*

   - ![截图：注册页面](assets/sign-up.png)
   - ![截图：Tavily API Keys 控制台](assets/api-key.png)


2. **注册** OpenAI 以获取你的 API key。默认情况下我们将使用 OpenAI——但你也可以替换为任何其他 LLM 提供商。


2. 从你的 Tavily 和 OpenAI 账号控制台**复制你的 API key**。

3. 将你的 API key**粘贴**到下面的单元格中并执行该单元格。


In [ ]:
# To export your API keys into a .env file, run the following cell (replace with your actual keys):
!echo "TAVILY_API_KEY=<your-tavily-api-key>" >> .env
!echo "OPENAI_API_KEY=<your-openai-api-key>" >> .env

在下面的单元格中安装依赖。

In [ ]:
%pip install -U tavily-python langchain-openai langchain langchain-tavily langgraph --quiet

### 配置你的 Tavily API 客户端

下面的代码将使用你的 API key 实例化 Tavily 客户端。

In [ ]:
import os
import getpass
from dotenv import load_dotenv
from tavily import TavilyClient

# Load environment variables from .env file
load_dotenv()

# Prompt the user to securely input the API key if not already set in the environment
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY:\n")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

# Initialize the Tavily API client using the loaded or provided API key
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

让我们使用 Tavily-LangChain 集成来定义以下模块化工具：
1. **搜索（Search）** Web 上的相关信息

2. **提取（Extract）** 特定网页的内容

3. **抓取（Crawl）** 整个网站

In [ ]:
# Define the set of web tools our agent will use to interact with the Tavily API.
from langchain_tavily import TavilySearch
from langchain_tavily import TavilyExtract
from langchain_tavily import TavilyCrawl

# Define the LangChain search tool
search = TavilySearch(max_results=10, topic="general")

# Define the LangChain extract tool
extract = TavilyExtract(extract_depth="advanced")

# Define the LangChain crawl tool
crawl = TavilyCrawl()

现在让我们配置几个 OpenAI 基础模型来驱动我们的 Agent，例如 o3-mini 和 gpt-4.1 模型。如果你更喜欢其他 LLM 提供商，可以轻松接入任何 LangChain Chat Model。

In [ ]:
# Instantiate the OpenAI foundation models
from langchain_openai import ChatOpenAI

# o3-mini-2025-01-31
o3_mini = ChatOpenAI(model="o3-mini-2025-01-31", api_key=os.getenv("OPENAI_API_KEY"))

# gpt-4.1
gpt_4_1 = ChatOpenAI(model="gpt-4.1", api_key=os.getenv("OPENAI_API_KEY"))

## Web Agent

接下来，我们将构建一个由 Tavily 驱动的 Web Agent，它由三个主要组件组成：语言模型、一组 Web 工具，以及系统提示词。语言模型（如 o3-mini 或 gpt-4.1）充当 Agent 的"大脑"，而 Web 工具（Search、Extract 和 Crawl）让 Agent 能够与互联网交互并从中收集信息。系统提示词引导 Agent 的行为，解释如何以及何时使用每个工具来完成其研究目标。

该 Agent 利用了预构建的 LangGraph reAct 实现，如下图所示。reAct 框架使 Agent 能够推理要采取哪些行动、按顺序使用 Web 工具，并在完成研究任务前按需迭代。系统提示词尤为重要——它指导 Agent 结合使用这些工具的最佳实践，确保 Agent 的回复全面、准确且有可靠的来源。

我们鼓励你试验系统提示词，或尝试不同的语言模型（比如在 gpt-4.1 和 o3-mini 之间切换），以改变 Agent 的风格、个性，或针对特定用例优化其性能。

<img src="./assets/web-agent.svg" alt="Agent" width="500"/>


In [ ]:
import datetime
from langgraph.prebuilt import create_react_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

today = datetime.datetime.today().strftime("%A, %B %d, %Y")

# Create the web agent
web_agent = create_react_agent(
    model=gpt_4_1,
    tools=[search, extract, crawl],
    prompt=ChatPromptTemplate.from_messages(
        [
            (
                "system",
                f"""    
        You are a research agent equipped with advanced web tools: Tavily Web Search, Web Crawl, and Web Extract. Your mission is to conduct comprehensive, accurate, and up-to-date research, grounding your findings in credible web sources.

        **Today's Date:** {today}

        **Available Tools:**

        1. **Tavily Web Search**

        * **Purpose:** Retrieve relevant web pages based on a query.
        * **Usage:** Provide a search query to receive semantically ranked results, each containing the title, URL, and a content snippet.
        * **Best Practices:**

            * Use specific queries to narrow down results.
            * Optimize searches using parameters such as `search_depth`, `time_range`, `include_domains`, and `include_raw_content`.
            * Break down complex queries into specific, focused sub-queries.

        2. **Tavily Web Crawl**

        * **Purpose:** Explore a website's structure and gather content from linked pages for deep research and information discovery from a single source.
        * **Usage:** Input a base URL to crawl, specifying parameters such as `max_depth`, `max_breadth`, and `extract_depth`.
        * **Best Practices:**

            * Begin with shallow crawls and progressively increase depth.
            * Utilize `select_paths` or `exclude_paths` to focus the crawl.
            * Set `extract_depth` to "advanced" for comprehensive extraction.

        3. **Tavily Web Extract**

        * **Purpose:** Extract the full content from specific web pages.
        * **Usage:** Provide URLs to retrieve detailed content.
        * **Best Practices:**

            * Set `extract_depth` to "advanced" for detailed content, including tables and embedded media.
            * Enable `include_images` if image data is necessary.

        **Guidelines for Conducting Research:**

        * **Citations:** Always support findings with source URLs, clearly provided as in-text citations.
        * **Accuracy:** Rely solely on data obtained via provided tools—never fabricate information.
        * **Methodology:** Follow a structured approach:

        * **Thought:** Consider necessary information and next steps.
        * **Action:** Select and execute appropriate tools.
        * **Observation:** Analyze obtained results.
        * Repeat Thought/Action/Observation cycles as needed.
        * **Final Answer:** Synthesize and present findings with citations in markdown format.

        **Example Workflows:**

        **Workflow 1: Search Only**

        **Question:** What are recent news headlines about artificial intelligence?

        * **Thought:** I need quick, recent articles about AI.
        * **Action:** Use Tavily Web Search with the query "recent artificial intelligence news" and set `time_range` to "week".
        * **Observation:** Retrieved 10 relevant articles from reputable news sources.
        * **Final Answer:** Summarize key headlines with citations.

        **Workflow 2: Search and Extract**

        **Question:** Provide detailed insights into recent advancements in quantum computing.

        * **Thought:** I should find recent detailed articles first.
        * **Action:** Use Tavily Web Search with the query "recent advancements in quantum computing" and set `time_range` to "month".
        * **Observation:** Retrieved 10 relevant results.
        * **Thought:** I should extract content from the most comprehensive article.
        * **Action:** Use Tavily Web Extract on the most relevant URL from search results.
        * **Observation:** Extracted detailed information about quantum computing advancements.
        * **Final Answer:** Provide detailed insights summarized from extracted content with citations.

        **Workflow 3: Search and Crawl**

        **Question:** What are the latest advancements in renewable energy technologies?

        * **Thought:** I need recent articles about advancements in renewable energy.
        * **Action:** Use Tavily Web Search with the query "latest advancements in renewable energy technologies" and set `time_range` to "month".
        * **Observation:** Retrieved 10 articles discussing recent developments in solar panels, wind turbines, and energy storage.
        * **Thought:** To gain deeper insights, I'll crawl a relevant industry-leading renewable energy site.
        * **Action:** Use Tavily Web Crawl on the URL of a leading renewable energy industry website, setting `max_depth` to 2.
        * **Observation:** Gathered extensive content from multiple articles linked on the site, highlighting new technologies and innovations.
        * **Final Answer:** Provide a synthesized summary of findings with citations.

        ---

        You will now receive a research question from the user:

        """,
            ),
            MessagesPlaceholder(variable_name="messages"),
        ]
    ),
    name="web_agent",
)

### 测试你的 Tavily Web Agent

现在我们将运行 Agent，看看它是如何使用不同 Web 工具的。

In [ ]:
from langchain_core.messages import HumanMessage

# Test the web agent
inputs = {
    "messages": [
        HumanMessage(
            content="find all the iphone models currently available on apple.com and their prices"
        )
    ]
}

# Stream the web agent's response
for s in web_agent.stream(inputs, stream_mode="values"):
    message = s["messages"][-1]
    if isinstance(message, tuple):
        print(message)
    else:
        message.pretty_print()

检查上方打印的 Agent 中间步骤，包括它如何选择和配置不同的工具参数。然后，以 markdown 格式显示 Agent 的最终回答。

In [ ]:
from IPython.display import Markdown

Markdown(message.content)

让我们尝试一个不同的例子。

In [ ]:
# Test the web agent
inputs = {
    "messages": [
        HumanMessage(
            content="return 5 job postings for a software engineer in the bay area on linkedin"
        )
    ]
}

# Stream the web agent's response
for s in web_agent.stream(inputs, stream_mode="values"):
    message = s["messages"][-1]
    if isinstance(message, tuple):
        print(message)
    else:
        message.pretty_print()

将 Agent 的最终回复打印为 markdown。

In [ ]:
# Print the agent's final response as markdown
Markdown(message.content)

注意 Agent 是如何巧妙地组合 Tavily 的工具——search、crawl 和 extract——端到端完成任务的。

## 结论与后续步骤

在本教程中，你学会了如何：
- 使用 LangChain 配置 Tavily Web 工具（search、extract、crawl）
- 使用 LangGraph 的 `create_react_agent` 构建智能 Web 研究 Agent
- 为自主 Web 研究设计有效的系统提示词
- 用产品研究和求职等真实任务测试你的 Agent

你现在拥有一个功能完整的 Web 研究 Agent，它能够自主地组合搜索、提取和抓取，完成复杂的研究目标。

**准备好迎接更高级的能力了吗？**
继续学习**教程 #3：构建混合 Agent**，了解如何将 Web 研究与内部向量搜索集成。你将为任何内部与外部知识都重要的场景解锁新能力。

[👉 **继续到教程 #3**](./hybrid-agent-tutorial.ipynb)
